## ensure all imports can be accessed

In [1]:
import sys, pathlib, importlib.util, importlib

repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "tests"))

importlib.invalidate_caches()

## Load new and old data processing pipelines

In [2]:
from data_processors.pull_and_process_data import master_function
from data_processors.load_processed_data import master_cleaning_and_saving
from data_processors.data_splitter import DataSplitter

from baseline_code.data_processors.BN_pull_and_process_data import master_function as master_function_old
from baseline_code.data_processors.BN_load_processed_data import master_cleaning_and_saving as master_cleaning_and_saving_old
from baseline_code.data_processors.BN_data_splitter import DataSplitter as DataSplitterOld

Using device: cuda
Using device: cuda


## Create performance tracker

In [3]:
import time, os, psutil, resource, tracemalloc, traceback

class Perf:
    def __enter__(self):
        self.p = psutil.Process(os.getpid())
        self.t0 = time.perf_counter()
        self.m0 = self.p.memory_info().rss
        self.io0 = self.p.io_counters()
        tracemalloc.start()
        return self
    def __exit__(self, exc_type, exc, tb):
        t1 = time.perf_counter()
        m1 = self.p.memory_info().rss
        io1 = self.p.io_counters()
        current, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        peak_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        print(f"\nTime {t1-self.t0:.3f}s | RAM Δ {(m1-self.m0)/1e6:.1f} MB | PeakRSS {peak_rss} | "
              f"PyPeak {(peak)/1e6:.1f} MB | "
              f"I/O Δ R {(io1.read_bytes-self.io0.read_bytes)/1e6:.1f} MB, "
              f"W {(io1.write_bytes-self.io0.write_bytes)/1e6:.1f} MB\n")
        return False  # propagate exceptions

## Baseline 'pull_and_process_data.py' performance test:

In [4]:
with Perf():
    mouse_number = 719161530
    spike_df = master_function_old(session_number=mouse_number, output_dir="/proj/STOR/pipiras/Neuropixel/Neuropixels-Pipeline-Refactor/src/output", timesteps_per_frame=10)

Using device: cuda
Updated version 3!
Initializing workflow...
Creating directory and manifest path...
Using existing manifest.json file.
Initializing EcephysProjectCache and session table...


/nas/longleaf/home/dkofma60/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:583: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.2.2, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


Fetching session data and spike times...


/nas/longleaf/home/dkofma60/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:583: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.2.2, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)
/nas/longleaf/home/dkofma60/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:583: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.2.2, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)
/nas/longleaf/home/dkofma60/.l

Session objects
['DETAILED_STIMULUS_PARAMETERS', 'LazyProperty', 'age_in_days', 'api', 'channel_structure_intervals', 'channels', 'conditionwise_spike_statistics', 'ecephys_session_id', 'from_nwb_path', 'full_genotype', 'get_current_source_density', 'get_inter_presentation_intervals_for_stimulus', 'get_invalid_times', 'get_lfp', 'get_parameter_values_for_stimulus', 'get_pupil_data', 'get_screen_gaze_data', 'get_stimulus_epochs', 'get_stimulus_parameter_values', 'get_stimulus_table', 'inter_presentation_intervals', 'invalid_times', 'mean_waveforms', 'metadata', 'num_channels', 'num_probes', 'num_stimulus_presentations', 'num_units', 'optogenetic_stimulation_epochs', 'presentationwise_spike_counts', 'presentationwise_spike_times', 'probes', 'rig_equipment_name', 'rig_geometry_data', 'running_speed', 'session_start_time', 'session_type', 'sex', 'specimen_name', 'spike_amplitudes', 'spike_times', 'stimulus_conditions', 'stimulus_names', 'stimulus_presentations', 'structure_acronyms', 'stru

...

Calculating bins and processing neurons...


Processing neurons batch 1: 100%|██████████| 1000/1000 [20:03<00:00,  1.20s/it] 


Memory usage: 8281.34 MB
GPU memory allocated: 0.41 GB
GPU memory cached: 0.52 GB


Processing neurons batch 2: 100%|██████████| 785/785 [15:30<00:00,  1.19s/it]

Memory usage: 8300.29 MB
GPU memory allocated: 0.59 GB
GPU memory cached: 0.75 GB

Time 2411.713s | RAM Δ 1377.2 MB | PeakRSS 24872720 | PyPeak 24710.8 MB | I/O Δ R 3039.2 MB, W 0.0 MB



## Refactored 'pull_and_process_data.py' performance test:

In [5]:
with Perf():
    mouse_number = 719161530
    spike_df = master_function(session_number=mouse_number, output_dir="/proj/STOR/pipiras/Neuropixel/Neuropixels-Pipeline-Refactor/src/output", timesteps_per_frame=10)

Using device: cuda
Updated version 3!
Initializing workflow...
Creating directory and manifest path...
Using existing manifest.json file.
Initializing EcephysProjectCache and session table...
Fetching session data and spike times...
Session objects
['DETAILED_STIMULUS_PARAMETERS', 'LazyProperty', 'age_in_days', 'api', 'channel_structure_intervals', 'channels', 'conditionwise_spike_statistics', 'ecephys_session_id', 'from_nwb_path', 'full_genotype', 'get_current_source_density', 'get_inter_presentation_intervals_for_stimulus', 'get_invalid_times', 'get_lfp', 'get_parameter_values_for_stimulus', 'get_pupil_data', 'get_screen_gaze_data', 'get_stimulus_epochs', 'get_stimulus_parameter_values', 'get_stimulus_table', 'inter_presentation_intervals', 'invalid_times', 'mean_waveforms', 'metadata', 'num_channels', 'num_probes', 'num_stimulus_presentations', 'num_units', 'optogenetic_stimulation_epochs', 'presentationwise_spike_counts', 'presentationwise_spike_times', 'probes', 'rig_equipment_nam

Filtering valid spike times: 100%|██████████| 1785/1785 [00:00<00:00, 2160.40it/s]


Calculating bins and processing neurons...


Processing neurons batch 1: 100%|██████████| 1000/1000 [06:45<00:00,  2.47it/s]


Memory usage: 8555.73 MB
GPU memory allocated: 0.24 GB
GPU memory cached: 0.31 GB


Processing neurons batch 2: 100%|██████████| 785/785 [05:16<00:00,  2.48it/s]


Memory usage: 8733.98 MB
GPU memory allocated: 0.19 GB
GPU memory cached: 0.24 GB
Spike matrix size -> (1785,)


/nas/longleaf/home/dkofma60/.local/lib/python3.11/site-packages/numpy/core/fromnumeric.py:2009: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  result = asarray(a).shape


Preparing spike DataFrame...
Save raw rate dataframes as a pickle file.
Total time elapsed: 1011.36 seconds

Time 1011.369s | RAM Δ 7478.9 MB | PeakRSS 26143004 | PyPeak 24709.2 MB | I/O Δ R 3038.0 MB, W 425.4 MB



## Baseline 'load_processed_data.py' performance test:

In [12]:
with Perf():
    master_cleaning_and_saving_old(session_id=719161530, output_dir='/proj/STOR/pipiras/Neuropixel/Neuropixels-Pipeline-Refactor/src/output', timesteps_per_frame=10, original_pickle_prefix='spike_trains_with_stimulus_session', new_pickle_prefix='normalized_firing_rates')

Data saved to /proj/STOR/pipiras/Neuropixel/Neuropixels-Pipeline-Refactor/src/output/normalized_firing_rates_719161530.pkl
Original data file for session 719161530 has been cleaned and saved.

Time 5.399s | RAM Δ 0.0 MB | PeakRSS 24883408 | PyPeak 2551.1 MB | I/O Δ R 0.0 MB, W 833.7 MB



## Refactored 'load_processed_data.py' performance test:

In [13]:
with Perf():
    master_cleaning_and_saving(session_id=719161530, original_pickle_prefix='spike_trains_with_stimulus_session', timesteps_per_frame=10)

Data saved to /proj/STOR/pipiras/Neuropixel/Neuropixels-Pipeline-Refactor/src/output/normalized_firing_rates_719161530_10.pkl
Original data file for session 719161530 has been cleaned and saved.

Time 5.204s | RAM Δ 0.0 MB | PeakRSS 24883408 | PyPeak 2551.1 MB | I/O Δ R 0.0 MB, W 833.7 MB



## Baseline 'data_splitter.py' performance test:

In [6]:
with Perf():
    new_splitter = DataSplitterOld(spike_df)

X_val shape: torch.Size([476, 10, 1780, 1])
y_val shape: torch.Size([476, 1])
 
X (array): Matrix of number of batch_size, time_steps_per_frame, num_nodes, and number of features per node.
X.shape = (B, T, N, F)
X_train shape: torch.Size([4284, 10, 1780, 1])
X_test shape: torch.Size([1190, 10, 1780, 1])
X_train type: <class 'torch.Tensor'>
 
X_val shape: torch.Size([476, 10, 1780, 1])
y_val shape: torch.Size([476, 1])
 
y_shape = [batch_size, unique_frames_shown_per_10_timesteps]
y_train shape: torch.Size([4284, 1])
y_test shape: torch.Size([1190, 1])
y_test type: <class 'torch.Tensor'>
 

Time 0.665s | RAM Δ -0.9 MB | PeakRSS 26143004 | PyPeak 1275.1 MB | I/O Δ R 0.0 MB, W 0.0 MB



## Refactored 'data_splitter.py' performance test:

In [7]:
with Perf():
    new_splitter = DataSplitter(spike_df)

X_val shape: torch.Size([476, 10, 1780])
y_val shape: torch.Size([476])
 
X (array): Matrix of number of batch_size, time_steps_per_frame, num_nodes, and number of features per node.
X.shape = (B, T, N, F)
X_train shape: torch.Size([4284, 10, 1780])
X_test shape: torch.Size([1190, 10, 1780])
X_train type: <class 'torch.Tensor'>
 
X_val shape: torch.Size([476, 10, 1780])
y_val shape: torch.Size([476])
 
y_shape = [batch_size, unique_frames_shown_per_10_timesteps]
y_train shape: torch.Size([4284])
y_test shape: torch.Size([1190])
y_test type: <class 'torch.Tensor'>
 

Time 0.626s | RAM Δ 424.2 MB | PeakRSS 26143004 | PyPeak 1275.1 MB | I/O Δ R 0.0 MB, W 0.0 MB

